# 05 — Forward Propagation, Logits, Softmax, and Prediction (Manual NumPy)


> **Learning contract.** Every code cell is preceded by an explanation of what the code does, why the operation exists mathematically, what tensor/array shapes are expected, and what production or business failure it prevents. Run the notebooks in numerical order in a fresh Conda environment.


Forward propagation is deterministic given inputs and parameters. For our MLP: $Z_1=XW_1+b_1$, $A_1=ReLU(Z_1)$, $Z_2=A_1W_2+b_2$. $Z_2$ contains **logits**, not probabilities. Softmax converts a vector of unbounded class scores into positive values summing to one.

$softmax(z_i)=\frac{e^{z_i}}{\sum_j e^{z_j}}$. The predicted class is `argmax(logits)`; softmax does not change the argmax.


## Code walkthrough — trace four real MNIST samples through the untrained network
The code prints shapes at each stage and shows the final probability matrix. At initialization, predictions are mostly arbitrary. That is expected: architecture supplies capacity, not knowledge.


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import urllib.request
import numpy as np
from sklearn.model_selection import train_test_split

SEED = 42
np.random.seed(SEED)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT / "data"
ARTIFACT_DIR = ROOT / "artifacts"
DATA_DIR.mkdir(exist_ok=True)
ARTIFACT_DIR.mkdir(exist_ok=True)
MNIST_PATH = DATA_DIR / "mnist.npz"
MNIST_URL = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz"


def load_official_mnist():
    if not MNIST_PATH.exists():
        print("Downloading official MNIST archive to", MNIST_PATH)
        urllib.request.urlretrieve(MNIST_URL, MNIST_PATH)
    with np.load(MNIST_PATH) as data:
        return (data["x_train"], data["y_train"], data["x_test"], data["y_test"])


def balanced_subset(x, y, per_class, seed=SEED):
    rng = np.random.default_rng(seed)
    selected = []
    for cls in range(10):
        candidates = np.flatnonzero(y == cls)
        selected.extend(rng.choice(candidates, size=per_class, replace=False))
    selected = np.asarray(selected)
    rng.shuffle(selected)
    return (x[selected], y[selected])


def prepare_splits():
    x_train_raw, y_train_raw, x_test_raw, y_test_raw = load_official_mnist()
    x_dev, y_dev = balanced_subset(x_train_raw, y_train_raw, per_class=600)
    x_test, y_test = balanced_subset(
        x_test_raw, y_test_raw, per_class=100, seed=SEED + 1
    )
    x_train, x_val, y_train, y_val = train_test_split(
        x_dev, y_dev, test_size=1000, random_state=SEED, stratify=y_dev
    )

    def transform(x):
        return x.reshape(len(x), -1).astype("float32") / 255.0

    return (
        transform(x_train),
        y_train,
        transform(x_val),
        y_val,
        transform(x_test),
        y_test,
    )


rng = np.random.default_rng(SEED)
W1 = rng.normal(0, np.sqrt(2 / 784), size=(784, 64)).astype("float32")
b1 = np.zeros((1, 64), dtype="float32")
W2 = rng.normal(0, np.sqrt(2 / 64), size=(64, 10)).astype("float32")
b2 = np.zeros((1, 10), dtype="float32")


def relu(z):
    return np.maximum(z, 0)


def softmax(z):
    shifted = z - z.max(axis=1, keepdims=True)
    exp = np.exp(shifted)
    return exp / exp.sum(axis=1, keepdims=True)


def forward(X):
    z1 = X @ W1 + b1
    a1 = relu(z1)
    logits = a1 @ W2 + b2
    return (z1, a1, logits)


X_train, y_train, X_val, y_val, X_test, y_test = prepare_splits()
z1, a1, logits = forward(X_train[:4])
probs = softmax(logits)
print("logits shape", logits.shape)
print("probability sums", probs.sum(axis=1))
print("true", y_train[:4])
print("pred", probs.argmax(axis=1))
print(np.round(probs, 3))

logits shape (4, 10)
probability sums [0.99999994 1.         1.         1.0000001 ]
true [0 1 9 5]
pred [9 9 9 5]
[[0.105 0.032 0.214 0.032 0.042 0.116 0.083 0.033 0.103 0.24 ]
 [0.135 0.078 0.081 0.069 0.111 0.095 0.076 0.13  0.088 0.137]
 [0.068 0.056 0.154 0.056 0.082 0.13  0.128 0.119 0.047 0.159]
 [0.099 0.075 0.078 0.095 0.087 0.151 0.113 0.122 0.087 0.094]]


## Code walkthrough — numerical stability of softmax
Exponentials can overflow when logits are large. Subtracting the maximum logit leaves all probabilities unchanged because softmax is shift-invariant, while preventing huge exponentials.


In [2]:
z = np.array([[1200.0, 1198.0, 1180.0]])
shifted = z - z.max(axis=1, keepdims=True)
p = np.exp(shifted) / np.exp(shifted).sum(axis=1, keepdims=True)
print("shifted logits", shifted)
print("stable softmax", p, "sum=", p.sum())

shifted logits [[  0.  -2. -20.]]
stable softmax [[8.80797076e-01 1.19202922e-01 1.81545808e-09]] sum= 1.0


## Business implication
A logit is model evidence, not a calibrated probability and not a business action. Production systems typically add calibration, thresholds, abstention/manual-review policies and downstream cost rules after the model score.


### Interactive Plotly lab — clone + run locally

## Interactive softmax confidence and temperature

> **GitHub vs local behavior.** GitHub keeps the committed static plots, tables, metrics and explanations visible. **Clone this branch, create `environment.yml`, open the notebook in VS Code/Jupyter, select the Conda kernel, and run the cell below for full Plotly interactivity**: hover, zoom, pan, 3D rotation, legend selection, sliders and animation.

The logits stay fixed while temperature changes. Class ranking can remain unchanged while probability sharpness changes substantially, which matters for confidence interpretation, calibration and decision thresholds.

**Production/business interpretation.** The interaction is an inspection instrument, not decoration: change or inspect a technical quantity and connect it to model behavior, operational risk or downstream decision quality.


In [3]:
import os
import numpy as np
import plotly.graph_objects as go

logits_demo = np.array([3.2, 1.7, 0.8, -0.4, 2.1])
temperatures = [0.25, 0.5, 1.0, 2.0, 4.0]
def softmax_t(logits, temperature):
    scaled = logits / temperature; exp = np.exp(scaled - scaled.max()); return exp / exp.sum()
probabilities = [softmax_t(logits_demo, t) for t in temperatures]
classes = [f"class {i}" for i in range(len(logits_demo))]
fig = go.Figure(data=[go.Bar(x=classes, y=probabilities[0], hovertemplate="%{x}<br>p=%{y:.4f}<extra></extra>")], frames=[go.Frame(name=str(i), data=[go.Bar(x=classes, y=p)]) for i, p in enumerate(probabilities)])
fig.update_layout(title="Softmax temperature: same logits, different confidence", yaxis=dict(title="probability", range=[0, 1]), sliders=[dict(currentvalue=dict(prefix="temperature = "), steps=[dict(label=str(t), method="animate", args=[[str(i)], dict(mode="immediate", frame=dict(duration=0), transition=dict(duration=0))]) for i, t in enumerate(temperatures)])])
if os.getenv("GITHUB_ACTIONS") == "true":
    print("Interactive softmax-temperature explorer built. Run locally to move the confidence slider.")
else:
    fig.show(renderer="plotly_mimetype")


Interactive softmax-temperature explorer built. Run locally to move the confidence slider.
